# UseItUp — Explainable Recipe Recommendation

**CS 4580/5580 Final Project** | Joseph Yu · Gavin Onghai · Helen Mao

---

UseItUp turns your pantry into a personalised meal plan and explains every decision in plain
language. Enter the ingredients you have on hand, choose your dietary goals and time budget,
then press **Recommend**. The system runs three stages:

1. **Stage 1 — Matching & Filtering:** scores every recipe by ingredient overlap and eliminates
   those that violate your hard dietary constraints (allergies, dietary restrictions).
2. **Stage 2 — Case-Based Reasoning:** retrieves the most similar recipe to your highest-rated
   past meals and adapts it to meet your current goals via ingredient substitution.
3. **Stage 3 — Explanation:** generates a goal trace, a counterfactual, a CBR lineage trace,
   and an ingredient-utilisation report.

Every cell is **idempotent** — re-run any cell at any time and the single `state` dict keeps
everything consistent. Work through cells in order on first use.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML, clear_output

# ── Locate project root (works from notebooks/ or project root) ──────────────
_here = Path().resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from useitup.pipeline import recommend, Recommendation
from useitup.profile import load_profile, SoftPreferences
from useitup.data_loader import load_recipes
from useitup.explain import render_explanation

# ── Load data ────────────────────────────────────────────────────────────────
RECIPES      = load_recipes(_root / "data" / "recipes_sample.json")
BASE_PROFILE = load_profile("demo_user", base_dir=_root / "data" / "profiles")

# ── Single mutable state dict ────────────────────────────────────────────────
state: dict = {
    "profile":   BASE_PROFILE,
    "results":   None,
    "out_main":  widgets.Output(),
    "out_trace": widgets.Output(),
    "out_alt":   widgets.Output(),
}

# ── Rendering helpers ────────────────────────────────────────────────────────
def _recipe_card_md(rec) -> str:
    stars = "★" * rec.difficulty + "☆" * (5 - rec.difficulty)
    header = (
        f"## 🍽 {rec.name}\n\n"
        f"**Cuisine:** {rec.cuisine} &nbsp;·&nbsp; "
        f"**Prep:** {rec.prep_time_min} min &nbsp;·&nbsp; "
        f"**Difficulty:** {stars}\n"
    )
    ing_lines = "\n".join(
        "- " + i.name + (f" — {i.quantity} {i.unit}".rstrip() if i.quantity else "")
        for i in rec.ingredients
    )
    step_lines = "\n".join(f"{n}. {s}" for n, s in enumerate(rec.instructions, 1))
    return f"{header}\n### Ingredients\n{ing_lines}\n\n### Instructions\n{step_lines}"


def _refresh_outputs(results=None) -> None:
    """Populate all three Output widgets from results (or state['results'])."""
    if results is None:
        results = state["results"]
    if not results:
        return
    r: Recommendation = results[0]
    recipe = r.adapted_recipe.recipe
    soft_map = {sr.recipe.id: sr.soft_score for sr in r.filter_result.survivors}

    # ── Recipe card + explanation ─────────────────────────────────────────────
    with state["out_main"]:
        clear_output(wait=True)
        display(Markdown(_recipe_card_md(recipe)))
        display(Markdown("---\n" + render_explanation(r.explanation)))

    # ── Decision trace bar chart + top-5 table ────────────────────────────────
    with state["out_trace"]:
        clear_output(wait=True)
        elim = Counter(e.rule_name for e in r.decision_log if not e.passed)
        if elim:
            fig, ax = plt.subplots(figsize=(7, 3))
            rules  = list(elim.keys())
            counts = [elim[k] for k in rules]
            colors = [
                "#c0392b"
                if any(kw in k for kw in ("Allergy", "Dietary", "Pantry")) else "#e67e22"
                for k in rules
            ]
            ax.barh(rules, counts, color=colors, edgecolor="white")
            ax.set_xlabel("Recipes Eliminated")
            ax.set_title("Stage 1 — Decision Log: Eliminations per Rule")
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            plt.tight_layout()
            plt.show()
            plt.close()
        rows = [
            {
                "Recipe":     m.recipe.name,
                "Cuisine":    m.recipe.cuisine,
                "CBR Sim":    round(m.similarity_score, 3),
                "Soft Score": round(soft_map.get(m.recipe.id, 0.0), 3),
                "Prep (min)": m.recipe.prep_time_min,
            }
            for m in r.cbr_matches[:5]
        ]
        df = pd.DataFrame(rows)
        display(Markdown("### Top 5 CBR Candidates"))
        display(HTML(df.to_html(index=False)))

    # ── Alt output placeholder (overwritten by scenario buttons) ─────────────
    with state["out_alt"]:
        clear_output(wait=True)
        display(Markdown(
            "*Click a scenario button in the **Alternative Scenarios** cell to see "
            "the counterfactual result here.*"
        ))


# ── Initial recommendation run ────────────────────────────────────────────────
state["results"] = recommend(state["profile"], RECIPES, top_k=5)
_refresh_outputs()
print(f"✓ {len(RECIPES)} recipes loaded.")
print(f"  Top pick: {state['results'][0].adapted_recipe.recipe.name!r}")

In [ ]:
# ── Pantry items ─────────────────────────────────────────────────────────────
pantry_box = widgets.Textarea(
    value=", ".join(BASE_PROFILE.pantry),
    placeholder="Comma-separated items, e.g.: eggs, garlic, chicken breast",
    description="Pantry:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="92%", height="80px"),
)

# ── Goals (multi-select) ──────────────────────────────────────────────────────
_GOAL_OPTIONS = [
    "high_protein", "low_cost", "vegetarian", "vegan", "low_carb", "quick", "dairy_free",
]
goals_select = widgets.SelectMultiple(
    options=_GOAL_OPTIONS,
    value=list(BASE_PROFILE.soft_preferences.goals),
    description="Goals:",
    style={"description_width": "80px"},
    layout=widgets.Layout(height="140px", width="92%"),
)

# ── Max prep-time slider ──────────────────────────────────────────────────────
_default_prep = BASE_PROFILE.soft_preferences.max_prep_time_min or 45
prep_slider = widgets.IntSlider(
    value=_default_prep,
    min=10, max=120, step=5,
    description="Max prep (min):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="92%"),
    readout_format="d",
)

# ── Recommend button ──────────────────────────────────────────────────────────
btn_recommend = widgets.Button(
    description="  Recommend",
    button_style="success",
    icon="cutlery",
    layout=widgets.Layout(width="180px", height="36px"),
)
status_lbl = widgets.Label(value="")


def _on_recommend(_btn):
    status_lbl.value = "⏳ Running pipeline …"
    pantry = [p.strip() for p in pantry_box.value.split(",") if p.strip()]
    new_prefs = BASE_PROFILE.soft_preferences.model_copy(update={
        "goals": list(goals_select.value),
        "max_prep_time_min": prep_slider.value,
    })
    new_profile = BASE_PROFILE.model_copy(update={
        "pantry": pantry,
        "soft_preferences": new_prefs,
    })
    state["profile"] = new_profile
    try:
        state["results"] = recommend(new_profile, RECIPES, top_k=5)
        _refresh_outputs()
        name = state["results"][0].adapted_recipe.recipe.name
        status_lbl.value = f"✓ Recommended: {name!r}"
    except ValueError as exc:
        status_lbl.value = f"✗ {exc}"


btn_recommend.on_click(_on_recommend)

display(widgets.VBox([
    widgets.Label("Select pantry ingredients, goals, and max prep time, then click Recommend:"),
    pantry_box,
    goals_select,
    prep_slider,
    widgets.HBox([btn_recommend, status_lbl]),
]))

In [ ]:
# Recipe card + full explanation — updates on every Recommend click.
# Re-running this cell re-displays the current recommendation.
display(state["out_main"])

In [ ]:
# Decision-log bar chart (Stage 1 eliminations) + top-5 CBR candidates table.
# Re-running re-displays the current trace.
display(state["out_trace"])

In [ ]:
# ── Helper ───────────────────────────────────────────────────────────────────
def _run_scenario(label: str, hc_override=None, prefs_kwargs=None) -> None:
    """Re-run the pipeline with a modified profile; show result in out_alt."""
    hc = hc_override if hc_override is not None else state["profile"].hard_constraints
    extra = prefs_kwargs or {}
    new_prefs = state["profile"].soft_preferences.model_copy(update=extra)
    new_profile = state["profile"].model_copy(update={
        "hard_constraints": hc,
        "soft_preferences": new_prefs,
    })
    with state["out_alt"]:
        clear_output(wait=True)
        display(Markdown(f"### ↔ Scenario: {label}"))
        try:
            alt = recommend(new_profile, RECIPES, top_k=5)
            r = alt[0]
            display(Markdown(_recipe_card_md(r.adapted_recipe.recipe)))
            display(Markdown("---\n" + render_explanation(r.explanation)))
        except ValueError as exc:
            display(Markdown(f"**No survivors after relaxing constraints:** {exc}"))


# ── Scenario buttons ──────────────────────────────────────────────────────────
btn_no_gluten = widgets.Button(
    description="What if I wasn't gluten-free?",
    button_style="info",
    layout=widgets.Layout(width="270px"),
)
btn_2h = widgets.Button(
    description="What if I had 2 hours to prep?",
    button_style="info",
    layout=widgets.Layout(width="270px"),
)


def _on_no_gluten(_btn):
    hc = [c for c in state["profile"].hard_constraints if c != "gluten-free"]
    _run_scenario("No gluten-free constraint", hc_override=hc)


def _on_2h(_btn):
    _run_scenario("2-hour prep window", prefs_kwargs={"max_prep_time_min": 120})


btn_no_gluten.on_click(_on_no_gluten)
btn_2h.on_click(_on_2h)

display(widgets.VBox([
    widgets.Label("Live counterfactual demos — modify the profile on the fly and re-run:"),
    widgets.HBox([btn_no_gluten, btn_2h]),
    state["out_alt"],
]))